# 05l - Rollout-aware GraphGRU vs ConvGRU canary

Confronto atomico e parameter-matched dopo il ritiro della ricorrenza H2. Usa soltanto episodi del train originale, con ruoli fit/calibration/development disgiunti. Nessun accesso agli split di test o al fresh test sigillato.

In [ ]:
import importlib,hashlib,json,os,shutil,subprocess,sys,time,zipfile
from pathlib import Path
ELM_REPOSITORY='https://github.com/Zagred47/giada.git';ELM_REF=os.environ.get('HAYFLOW_ELM_REF','main');ROOT=Path('/kaggle/working');ELM_REPO=ROOT/'hayflow_workspace'/'elmneuron';ELM_REPO.parent.mkdir(parents=True,exist_ok=True)
def run(command,cwd=None):print('+',' '.join(map(str,command)),flush=True);subprocess.run(list(map(str,command)),cwd=cwd,check=True)
if not (ELM_REPO/'.git').is_dir():run(['git','clone',ELM_REPOSITORY,ELM_REPO])
run(['git','fetch','origin',ELM_REF],cwd=ELM_REPO);run(['git','checkout','--detach','FETCH_HEAD'],cwd=ELM_REPO);REVISION=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ELM_REPO,text=True).strip();sys.path.insert(0,str(ELM_REPO));importlib.invalidate_caches();print({'revision':REVISION})

## 1. Individua gli input esatti e prepara il dataset logico

In [ ]:
from src.hayflow_model.rollout_aware_architecture_canary import EXPECTED_05KD_INDEX_SHA256,discover_indexed_artifact_source
INPUT_ROOT=Path('/kaggle/input')
def extract_zip_safely(source,destination):
 source,destination=Path(source),Path(destination);marker=destination/'.source_size';stamp=str(source.stat().st_size)
 if marker.is_file() and marker.read_text().strip()==stamp:return destination
 if destination.exists():shutil.rmtree(destination)
 destination.mkdir(parents=True);root=destination.resolve()
 with zipfile.ZipFile(source) as archive:
  for member in archive.infolist():
   target=(destination/member.filename).resolve();assert target==root or root in target.parents,member.filename
  archive.extractall(destination)
 marker.write_text(stamp);return destination
decision_override=os.environ.get('HAYFLOW_05KD_ARTIFACT');ARTIFACT_05KD_SOURCE=discover_indexed_artifact_source(INPUT_ROOT,EXPECTED_05KD_INDEX_SHA256,override=Path(decision_override) if decision_override else None);assert ARTIFACT_05KD_SOURCE is not None,'Artefatto 05k-d esatto non trovato: aggiungi il Dataset Kaggle oppure imposta HAYFLOW_05KD_ARTIFACT alla cartella o allo ZIP (anche archive.zip).'
topup_override=os.environ.get('HAYFLOW_TOPUP_V3');topup_candidates=([Path(topup_override).expanduser()] if topup_override else [])+list(INPUT_ROOT.rglob('hayflow_bap_validation_support_topup_v3.zip'))+[p.parent for p in INPUT_ROOT.rglob('composite_dataset_manifest.json')]
TOPUP_SOURCE=next((p.resolve() for p in topup_candidates if p.exists()),None);assert TOPUP_SOURCE is not None,'Top-up BAP v3 non trovato.';TOPUP_ROOT=extract_zip_safely(TOPUP_SOURCE,'/kaggle/working/hayflow05l_topup') if TOPUP_SOURCE.is_file() else TOPUP_SOURCE
manifest_candidates=list(Path(TOPUP_ROOT).rglob('composite_dataset_manifest.json'));assert len(manifest_candidates)==1,manifest_candidates;COMPOSITE_MANIFEST=manifest_candidates[0]
base_override=os.environ.get('HAYFLOW_BASE_DATASET');base_candidates=([Path(base_override).expanduser()] if base_override else [])+[p.parent for p in INPUT_ROOT.rglob('transition_dataset.h5') if 'targeted' in str(p).lower() and 'topup' not in str(p).lower()]+[p for p in INPUT_ROOT.rglob('archive.zip') if 'hayflow-targeted-transition-dataset' in str(p).lower()]
BASE_SOURCE=next((p.resolve() for p in base_candidates if p.exists()),None);assert BASE_SOURCE is not None,'Dataset base targeted v1.1 non trovato.';print({'05k-d':str(ARTIFACT_05KD_SOURCE),'manifest':str(COMPOSITE_MANIFEST),'base':str(BASE_SOURCE)})

In [ ]:
from src.hayflow_data import prepare_composite_flowmap_bundle
hash_started={};hash_last={}
def hash_progress(name,done,total):
 now=time.monotonic();hash_started.setdefault(name,now);percent=int(100*done/total)
 if percent>=hash_last.get(name,-10)+10 or done==total:
  elapsed=now-hash_started[name];rate=done/max(elapsed,1e-9);eta=(total-done)/max(rate,1e-9);print(f'[HayFlow 05l][SHA-256 {name}] {percent}% ETA {eta/60:.1f} min',flush=True);hash_last[name]=percent
bundle=prepare_composite_flowmap_bundle(COMPOSITE_MANIFEST,base_source=BASE_SOURCE,progress=hash_progress);assert bundle.manifest['valid'] and bundle.transition_count==29880 and not bundle.manifest['physical_merge_performed'];print({'dataset_valid':True,'transitions':bundle.transition_count,'fingerprint':bundle.fingerprint})

## 2. Preflight e training closed-loop

La prima cella verifica isolamento dei ruoli, input causali e parita dei parametri. La seconda allena sei piccoli canary (due famiglie per tre seed), con tracker compatto ogni 10 epoche.

In [ ]:
import yaml
from IPython.display import display
from src.hayflow_model import RolloutAwareArchitectureCanary,RolloutAwareArchitectureCanaryConfig
cfg=yaml.safe_load((ELM_REPO/'configs/hayflow/hayflow_rollout_aware_architecture_canary.yml').read_text());config=RolloutAwareArchitectureCanaryConfig.from_mapping(cfg['rollout_aware_architecture_canary'])
OUTPUT_DIR=Path('/kaggle/working/artifacts/hayflow_rollout_aware_graphgru_vs_convgru_canary');assert not OUTPUT_DIR.exists(),f'Output gia presente: {OUTPUT_DIR}. Avvia una sessione nuova.'
session=RolloutAwareArchitectureCanary(bundle,OUTPUT_DIR,config,ARTIFACT_05KD_SOURCE,code_revision=REVISION);preflight=session.prepare();display({'valid':preflight['valid'],'episodes':preflight['role_episode_counts'],'windows':preflight['role_window_counts'],'parameters':preflight['parameter_counts'],'parameter_ratio':preflight['parameter_ratio'],'input_features':len(preflight['causal_drive_features']),'source_split':preflight['source_split_used']});assert preflight['valid']

In [ ]:
try:
 final_report=session.run()
finally:
 session.close()
display({'valid':final_report['valid'],'diagnosis':final_report['diagnosis'],'passing_seeds':final_report['passing_seed_count'],'median_rmse_8ms_mv':final_report['median_development_rmse_8ms_mv'],'graph_gain_vs_convgru':final_report['graph_improvement_vs_convgru_fraction'],'full_training':final_report['full_training_authorized'],'next_step':final_report['next_step']});assert final_report['valid'] and not final_report['full_training_authorized']

## 3. Crea e scarica lo ZIP con il downloader browser stabile

In [ ]:
from shutil import make_archive
import base64
from IPython.display import Javascript,display
zip_path=Path(make_archive('/kaggle/working/hayflow_rollout_aware_graphgru_vs_convgru_canary','zip',root_dir=OUTPUT_DIR.parent,base_dir=OUTPUT_DIR.name));payload=base64.b64encode(zip_path.read_bytes()).decode('ascii');filename=zip_path.name
display(Javascript(f"""const binary=atob('{payload}');const bytes=new Uint8Array(binary.length);for(let i=0;i<binary.length;i++)bytes[i]=binary.charCodeAt(i);const blob=new Blob([bytes],{{type:'application/zip'}});const url=URL.createObjectURL(blob);const a=document.createElement('a');a.href=url;a.download='{filename}';document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),60000);"""));print({'zip':str(zip_path),'size_mib':round(zip_path.stat().st_size/2**20,2),'download':'avviato dal browser'})